## Base model:

In [42]:
import pandas as pd
import numpy as np
import warnings 
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
from sklearn.model_selection import cross_val_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv("../data/feature_engineered_data.csv")
df

,person_age,person_income,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,person_education_Bachelor,person_education_Doctorate,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_status
0,22.0,71948.0,35000.0,16.02,0.49,3.0,561,0,0,0,0,0,1,0,0,0,1,0,1
1,21.0,12282.0,1000.0,11.14,0.08,2.0,504,1,0,0,0,1,0,1,0,0,0,0,0
2,25.0,12438.0,5500.0,12.87,0.44,3.0,635,0,0,0,0,0,0,0,0,1,0,0,1
3,23.0,79753.0,35000.0,15.23,0.44,2.0,675,0,1,0,0,0,1,0,0,1,0,0,1
4,24.0,66135.0,35000.0,14.27,0.53,4.0,586,0,0,0,0,0,1,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44988,27.0,47971.0,15000.0,15.66,0.31,3.0,645,0,0,0,0,0,1,0,0,1,0,0,1
44989,37.0,65800.0,9000.0,14.07,0.14,11.0,621,0,0,0,0,0,1,0,1,0,0,0,1
44990,33.0,56942.0,2771.0,10.02,0.05,10.0,668,0,0,0,0,0,1,0,0,0,0,0,1
44991,29.0,33164.0,12000.0,13.23,0.36,6.0,604,0,1,0,0,0,1,1,0,0,0,0,1


In [7]:
X = df.drop("loan_status", axis=1)
y = df["loan_status"]
numerical_cols = X.select_dtypes(include='number').columns

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42,stratify=y)
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

model = RandomForestClassifier(random_state=42)

score = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring='f1'   
)

print(score)
print("Mean F1:",score.mean())
print("Std:",score.std())

[0.82282681 0.83578104 0.83243968 0.83401084 0.81891348]
Mean F1: 0.8287943697789363
Std: 0.006671689039005118


## Experiment regarding n_estimator :

In [18]:
n_estimators_values = [50, 100, 200, 300, 500]
results = []
for n in n_estimators_values:
    model = RandomForestClassifier(
        random_state=42,
        n_estimators=n
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='f1'
    )

    results.append({
        "n_estimator" : n,
        "Mean f1" : score.mean(),
        "Std" : score.std()
    })

In [22]:
comparison = pd.DataFrame(results)
comparison.sort_values(
    by="Mean f1",
    ascending=False,
    inplace=True
)

print(comparison)

   n_estimator   Mean f1       Std
2          200  0.830143  0.007201
3          300  0.829500  0.007321
4          500  0.829377  0.008218
1          100  0.828794  0.006672
0           50  0.825126  0.008071


---->n_estimator = 200 is the best<----

## Experiment regarding max_depth :

In [24]:
max_depth_values = [5, 10, 15, 20, None]
results = []
for max_depth in max_depth_values:
    model = RandomForestClassifier(
        random_state=42,
        n_estimators=200,
        max_depth=max_depth
    )
    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='f1'
    )
    results.append({
        "max_depth" : max_depth,
        "Mean f1" : score.mean(),
        "Std" : score.std()
    })

In [25]:
comparison = pd.DataFrame(results)
comparison.sort_values(
    by='Mean f1',
    ascending=False,
    inplace=True
)
print(comparison)

   max_depth   Mean f1       Std
4        NaN  0.830143  0.007201
3       20.0  0.828796  0.006264
2       15.0  0.822432  0.003806
1       10.0  0.809831  0.003794
0        5.0  0.773293  0.003938


## Experiments regarding min_samples_split :

In [29]:
min_samples_split_values = [2, 5, 10, 20]
results = []
for min_sample_split in min_samples_split_values :
    model = RandomForestClassifier(
        random_state=42,
        n_estimators=200,
        max_depth=None,
        min_samples_split=min_sample_split
    )
    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='f1'
    )

    results.append({
        "min_sample_split":min_sample_split,
        "Mean f1" : score.mean(),
        "Std" : score.std()
    })

In [30]:
comparison = pd.DataFrame(results)
comparison.sort_values(
    by='Mean f1',
    ascending=False,
    inplace=True
)
print(comparison)

   min_sample_split   Mean f1       Std
0                 2  0.830143  0.007201
1                 5  0.826696  0.006719
2                10  0.825742  0.005155
3                20  0.822638  0.005039


## Experiment regarding min_samples_leaf :

In [37]:
min_samples_leaf_values = [1, 2, 4, 8]
results = []

for min_samples_leaf in min_samples_leaf_values :

    model = RandomForestClassifier(
        random_state=42,
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=min_samples_leaf
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='f1'
    )

    results.append({
        "min_sample_leaf":min_samples_leaf,
        "Mean f1" : score.mean(),
        "Std" : score.std()
    })
    


In [38]:
comparison = pd.DataFrame(results)
comparison.sort_values(
    by='Mean f1',
    ascending=False,
    inplace=True
)
print(comparison)

   min_sample_leaf   Mean f1       Std
0                1  0.830143  0.007201
1                2  0.826664  0.005222
2                4  0.822961  0.005667
3                8  0.819377  0.003725


## Experiment regarding max_features :

In [40]:
max_features_values = [
    "sqrt",
    "log2",
    None
]
results = []

for features in max_features_values :
      
    model = RandomForestClassifier(
        random_state=42,
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=features
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='f1'
    )

    results.append({
        "Max_features":features,
        "Mean f1" : score.mean(),
        "Std" : score.std()
    })

In [41]:
comparison = pd.DataFrame(results)
comparison.sort_values(
    by='Mean f1',
    ascending=False,
    inplace=True
)
print(comparison)

  Max_features   Mean f1       Std
0         sqrt  0.830143  0.007201
1         log2  0.830143  0.007201
2          NaN  0.828494  0.008279


### Using grid search cv

In [44]:
rf = RandomForestClassifier(random_state=42)
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)
best_model = grid_search.best_estimator_

Fitting 5 folds for each of 48 candidates, totalling 240 fits
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
0.8301427867275294
